In [0]:
dbutils.widgets.text("catalog", "claims_dev")

In [0]:
from pyspark.sql import functions as F

catalog = dbutils.widgets.get("catalog")
chk = f"/Volumes/{catalog}/landing/checkpoints"

text_cols = ["event_id", "claim_id", "policy_id", "event_type", "state", "city", "cause",
             "coverage_code", "workshop_id", "source", "channel", "status"]


def bronze_events(table):
    return spark.readStream.table(f"{catalog}.bronze.{table}").select(
        *[F.col(c).cast("string").alias(c) for c in text_cols],
        F.col("event_ts").cast("timestamp").alias("event_ts"),
        F.col("amount").cast("decimal(14,2)").alias("amount"),
        F.col("arrival_date").cast("date").alias("arrival_date"),
    )


policies = spark.table(f"{catalog}.silver.policies").select(
    "policy_id", "plan", "vehicle_value",
    F.col("__START_AT").alias("valid_from"),
    F.coalesce(F.col("__END_AT"), F.lit("9999-12-31").cast("timestamp")).alias("valid_to"),
)

In [0]:
events = (
    bronze_events("claim_events_app")
    .unionByName(bronze_events("claim_events_tpa"))
    .withWatermark("event_ts", "3 days")
    .dropDuplicatesWithinWatermark(["event_id"])
)

joined = events.alias("e").join(
    policies.alias("p"),
    (F.col("e.policy_id") == F.col("p.policy_id"))
    & (F.col("e.event_ts") >= F.col("p.valid_from"))
    & (F.col("e.event_ts") < F.col("p.valid_to")),
    "left",
).select(
    "e.*",
    F.col("p.plan"),
    F.col("p.vehicle_value"),
    F.col("p.policy_id").isNotNull().alias("policy_in_force"),
)

query = (
    joined.writeStream
    .option("checkpointLocation", f"{chk}/silver_claim_events")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.silver.claim_events")
)
query.awaitTermination()
spark.table(f"{catalog}.silver.claim_events").count()

In [0]:
hourly = (
    spark.readStream.table(f"{catalog}.silver.claim_events")
    .filter("event_type = 'FNOL'")
    .withWatermark("event_ts", "3 days")
    .groupBy(F.window("event_ts", "1 hour"), "city")
    .agg(F.count("*").alias("fnol_count"))
    .select(F.col("window.start").alias("hour_start"), "city", "fnol_count")
)

query = (
    hourly.writeStream
    .outputMode("append")
    .option("checkpointLocation", f"{chk}/silver_fnol_hourly")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.silver.fnol_hourly")
)
query.awaitTermination()

In [0]:
display(spark.sql(f"""
    SELECT count(*) AS events, count(DISTINCT event_id) AS unique_ids,
           count_if(NOT policy_in_force) AS without_policy
    FROM {catalog}.silver.claim_events
"""))
display(spark.sql(f"""
    SELECT hour_start, fnol_count FROM {catalog}.silver.fnol_hourly
    WHERE city = 'Ciudad de Mexico'
    ORDER BY fnol_count DESC
    LIMIT 5
"""))